# Research Paper RAG Chatbot (Pinecone Cloud Backend)

This notebook demonstrates indexing a research paper PDF into a Pinecone Serverless index and querying it with conversational memory.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_settings
from src.loader import load_pdf_chunks
from src.vectorstore import PineconeVectorStore
from src.rag import ResearchAssistant

## 1. Initialize Pinecone Vector Store

Ensure `GEMINI_API_KEY` and `PINECONE_API_KEY` are set in `.env`.

In [ ]:
settings = get_settings()
pdf_path = PROJECT_ROOT / "data" / "RAGPaper.pdf"

store = PineconeVectorStore(
    api_key=settings.pinecone_api_key,
    index_name=settings.pinecone_index_name,
    dimension=settings.embedding_dim,
    cloud=settings.pinecone_cloud,
    region=settings.pinecone_region,
)
assistant = ResearchAssistant(store=store, settings=settings)

## 2. Chunk and Upsert the Research Paper

In [ ]:
chunks = load_pdf_chunks(
    pdf_path,
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)
assistant.index_chunks(chunks)
print(f"Upserted {len(chunks)} chunks into Pinecone index '{settings.pinecone_index_name}'")

## 3. Query the Indexed Paper

In [ ]:
answer, matches = assistant.ask("Summarize the main findings and limitations discussed in the paper.")
print(answer)
print("\nCited pages:", sorted({c.page for c, _ in matches}))